In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import lightgbm as lgb

OSError: dlopen(/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/lightgbm/lib/lib_lightgbm.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib
  Referenced from: <D44045CD-B874-3A27-9A61-F131D99AACE4> /Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/lightgbm/lib/lib_lightgbm.dylib
  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/local/lib/libomp/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/local/lib/libomp/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/local/lib/libomp/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/local/lib/libomp/libomp.dylib' (no such file)

In [ ]:
def handle_missing_data(df):
    # Competitor cleanup
    inv_cols  = [f'comp{i}_inv' for i in range(1, 9)]
    rate_cols = [f'comp{i}_rate' for i in range(1, 9)]
    pct_cols  = [f'comp{i}_rate_percent_diff' for i in range(1, 9)]
    
    df[inv_cols] = df[inv_cols].fillna(0).replace(-1, 0)
    df[rate_cols] = df[rate_cols].fillna(0)
    for rate, pct in zip(rate_cols, pct_cols):
        df[pct] = df[pct].fillna(0) * df[rate]
    df.drop(columns=rate_cols, inplace=True)
    
    # Drop truly “too many missing” columns
    df.drop(columns=[
        'gross_bookings_usd',
        'srch_query_affinity_score',
        'prop_location_score2'
    ], errors='ignore', inplace=True)
    
    # Reviews
    df['prop_review_score'] = df['prop_review_score'].fillna(0)
    
    return df

def transform_dates_and_log_price(df):
    df = df.copy()
    df['date_time'] = pd.to_datetime(df['date_time'])
    # date parts
    df['month'] = df['date_time'].dt.month
    df['day']   = df['date_time'].dt.day
    # length_of_stay & booking_window
    df['srch_ci'] = pd.to_datetime(df['srch_ci'])
    df['srch_co'] = pd.to_datetime(df['srch_co'])
    df['length_of_stay'] = (df['srch_co'] - df['srch_ci']).dt.days
    df['booking_window'] = (df['srch_ci'] - df['date_time']).dt.days
    df.drop(columns=['date_time','srch_ci','srch_co'], inplace=True)
    
    # log transforms
    eps = 1e-5
    df['log_price_usd']            = np.log(df['price_usd'] + eps)
    df['log_visitor_hist_adr_usd'] = np.log(df['visitor_hist_adr_usd'].fillna(0) + eps)
    df.drop(columns=['price_usd','visitor_hist_adr_usd'], inplace=True)
    
    return df

def aggregator(df, groupby_cols, target_cols, aggregators):
    df_grouped = df.groupby(groupby_cols)[target_cols].agg(aggregators)
    flat_cols = []
    for col, func in df_grouped.columns:
        func_name = func if isinstance(func, str) else func.__name__
        flat_cols.append(f"{col}_{func_name}_by_{'_'.join(groupby_cols)}")
    df_grouped.columns = flat_cols
    return df.merge(df_grouped, on=groupby_cols, how='left')

# --- Processor class --------------------------------------------------------

class Processor:
    def __init__(self,
                 group_target_map,
                 aggregators,
                 features_to_minmax,
                 features_to_standardize,
                 categorical_features):
        self.group_target_map      = group_target_map
        self.aggregators           = aggregators
        self.features_to_minmax    = features_to_minmax
        self.features_to_standardize = features_to_standardize
        self.categorical_features  = categorical_features
        self.normalizer = MinMaxScaler()
        self.standardizer = StandardScaler()
    
    def process(self, df, training=True):
        df = df.copy()
        # 1) Outlier filter
        p_low, p_high = df['price_usd'].quantile([0.1, 0.9])
        d_low, d_high = df['orig_destination_distance'].quantile([0.1, 0.9])
        mask = (df['price_usd'].between(p_low, p_high) &
                df['orig_destination_distance'].between(d_low, d_high))
        df = df.loc[mask]
        
        # 2) Missing data & competitors
        df = handle_missing_data(df)
        
        # 3) Flags
        flag_cols = ['log_price_usd', 'prop_review_score',
                     'prop_brand_bool', 'prop_starrating',
                     'prop_location_score1', 'orig_destination_distance']
        for c in flag_cols:
            if c in df:
                df[f"{c}_missing"] = df[c].isnull().astype(int)
        
        # 4) Impute remaining nulls
        rng = np.random.default_rng(42)
        for c in ['visitor_hist_starrating','prop_review_score']:
            if c in df:
                vals = df[c].dropna().to_numpy()
                mask_missing = df[c].isna()
                df.loc[mask_missing, c] = rng.choice(vals, size=mask_missing.sum(), replace=True)
        df['orig_destination_distance'].fillna(-1, inplace=True)
        
        # 5) Date & log transformations
        df = transform_dates_and_log_price(df)
        
        # 6) Aggregations for each group
        for groupby_col, targets in self.group_target_map.items():
            df = aggregator(df, [groupby_col], targets, self.aggregators)
        
        df.fillna(0, inplace=True)
        
        # 7) Scaling
        if training:
            df[self.features_to_minmax]    = self.normalizer.fit_transform(df[self.features_to_minmax])
            df[self.features_to_standardize] = self.standardizer.fit_transform(df[self.features_to_standardize])
        else:
            df[self.features_to_minmax]    = self.normalizer.transform(df[self.features_to_minmax])
            df[self.features_to_standardize] = self.standardizer.transform(df[self.features_to_standardize])
        
        return df

# --- Main pipeline ----------------------------------------------------------

# 1) Load raw data
raw_path = "./datasets/training_set_VU_DM_reduced.csv"
df_raw = pd.read_csv(raw_path)

# 2) Competitor variable list
comp_vars = [c for c in df_raw.columns if c.startswith('comp') and 'rate' not in c]

# 3) Define target lists
srch_targets = ['log_price_usd','prop_starrating','prop_location_score1',
                'prop_review_score','prop_brand_bool'] + comp_vars
prop_targets = ['log_price_usd','prop_starrating','prop_brand_bool',
                'promotion_flag'] + comp_vars
dest_targets = ['log_price_usd','prop_starrating','prop_location_score1',
                'prop_review_score','prop_brand_bool'] + comp_vars

group_target_map = {
    'srch_id': srch_targets,
    'prop_id': prop_targets,
    'srch_destination_id': dest_targets
}

# 4) Define feature sets for scaling
features_minmax = ['prop_starrating','prop_review_score','prop_location_score1']
features_standard = (
    [c for c in srch_targets if c.startswith('log_')] +
    ['length_of_stay','booking_window'] +
    ['srch_adults_count','srch_children_count','srch_room_count'] +
    [c for c in comp_vars]
)

cat_features = ['site_id','posa_continent','month','day']

# 5) Process training data
processor = Processor(group_target_map,
                      aggregators=['mean','std'],
                      features_to_minmax=features_minmax,
                      features_to_standardize=features_standard,
                      categorical_features=cat_features)

df_processed = processor.process(df_raw, training=True)

# 6) Split to train/validation
label = 'booking_bool'
X = df_processed.drop(columns=[label])
y = df_processed[label]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 7) Prepare LightGBM datasets
train_group = X_train.groupby('srch_id').size().values
val_group   = X_val.groupby('srch_id').size().values

lgb_train = lgb.Dataset(X_train, label=y_train,
                        group=train_group,
                        categorical_feature=cat_features)

lgb_val   = lgb.Dataset(X_val, label=y_val, reference=lgb_train,
                        group=val_group,
                        categorical_feature=cat_features)

# 8) Train a LightGBM ranker
params = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'verbose': -1
}

model = lgb.train(params,
                  lgb_train,
                  valid_sets=[lgb_val],
                  early_stopping_rounds=50)



/var/folders/zd/05ndgbcd4k35_hn1kps4c8sr0000gn/T/ipykernel_27222/150152299.py:57: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['orig_destination_distance'].fillna(-1, inplace=True)


AttributeError: 'NoneType' object has no attribute 'groupby'

In [ ]:
comp_vars = [c for c in df.columns 
             if c.startswith('comp') and 'rate' not in c]
